电商数据中存在大量未完成的订单。如果不加筛选，直接算 GMV 或复购率，数据会被严重污染。
在 Olist 的订单表中，order_status 有以下几种状态：
· delivered（已送达）：正常完成，这是我们要的分析主体。
· shipped / processing / invoiced（运输中/处理中/已开票）：未完成，没有最终的配送时间和评分。
· canceled（已取消）：不计入销售。
· unavailable（不可用）：不计入销售。
决策：我们只保留 order_status == 'delivered' 的订单。
原因：只有已送达的订单，才能准确计算配送时长，并且有完整的用户评价，适合做体验分析。

评论表 olist_order_reviews_dataset.csv 中，同一个订单（order_id）可能有多条评论（用户追加评论）。如果直接和订单表连接，会导致订单被复制多行，数据膨胀。
决策：对于每个订单，只保留最新的一条评论。

In [2]:
import pandas as pd
import os

data_path = '../data/raw'
processed_path = '../data/processed'
os.makedirs(processed_path, exist_ok=True)

# 1. 读取订单表
orders = pd.read_csv(os.path.join(data_path, 'olist_orders_dataset.csv'))
print(f"原始订单表: {orders.shape}")

# 2. 只保留已送达订单
orders = orders[orders['order_status'] == 'delivered'].copy()
print(f"筛选已送达后: {orders.shape}")

# 3. 读取评论表并去重
# 读取评论表
reviews = pd.read_csv(os.path.join(data_path, 'olist_order_reviews_dataset.csv'))
print(f"原始评论表: {reviews.shape}")

# 按照 order_id 聚合
reviews_agg = reviews.groupby('order_id').agg(
    review_count=('review_id', 'count'),          # 该订单有几条评论
    review_score_mean=('review_score', 'mean'),   # 平均评分
    review_score_min=('review_score', 'min')      # 最低评分（反映最差体验）
).reset_index()

print(f"聚合后评论表: {reviews_agg.shape}")
print(reviews_agg.head())

原始订单表: (99441, 8)
筛选已送达后: (96478, 8)
原始评论表: (99224, 7)
聚合后评论表: (98673, 4)
                           order_id  review_count  review_score_mean  \
0  00010242fe8c5a6d1ba2dd792cb16214             1                5.0   
1  00018f77f2f0320c557190d7a144bdd3             1                4.0   
2  000229ec398224ef6ca0657da4fc703e             1                5.0   
3  00024acbcdf0a6daa1e931b038114c75             1                4.0   
4  00042b26cf59d7ce69dfabb4e55b4fd9             1                5.0   

   review_score_min  
0                 5  
1                 4  
2                 5  
3                 4  
4                 5  


In [3]:
# 需要转换的日期列
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# 查看转换后的数据类型和缺失情况
print(orders[date_columns].dtypes)
print("\n各日期列缺失值：")
print(orders[date_columns].isnull().sum())

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

各日期列缺失值：
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [4]:
# 实际配送天数：送达时间 - 下单时间
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

# 预计配送天数：预计送达时间 - 下单时间
orders['estimated_days'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.days

# 是否延迟：实际配送天数 > 预计配送天数
orders['is_late'] = orders['delivery_days'] > orders['estimated_days']

# 查看统计信息
print(orders[['delivery_days', 'estimated_days', 'is_late']].describe())
print(f"\n延迟订单比例: {orders['is_late'].mean():.2%}")

       delivery_days  estimated_days
count   96470.000000    96478.000000
mean       12.093604       23.372759
std         9.551380        8.758137
min         0.000000        2.000000
25%         6.000000       18.000000
50%        10.000000       23.000000
75%        15.000000       28.000000
max       209.000000      155.000000

延迟订单比例: 7.57%


· 缺失值极少：order_approved_at 缺 14 个，order_delivered_customer_date 缺 8 个。因为我们只保留了 delivered 状态的订单，所以这几个是数据异常记录，之后我们做分析时直接忽略这 8 条即可。
· 配送时间极长：delivery_days 最大 209 天，这通常是因为个别偏远地区或特殊商品（如大件家具），属于极端值，暂时保留。
· 延迟率 7.57%：这是一个非常真实的电商数据。注意看 estimated_days 均值是 23 天，而实际 delivery_days 均值是 12 天。平台给用户的承诺非常保守（预估 23 天），所以只有 7.57% 的订单会超过这个预估。

In [5]:
# 读取用户表
customers = pd.read_csv(os.path.join(data_path, 'olist_customers_dataset.csv'))

# 合并用户信息（customer_state, customer_city）
orders = orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']], 
    on='customer_id', 
    how='left'
)
print(f"合并用户后: {orders.shape}")

合并用户后: (96478, 14)


In [6]:
# 读取支付表
payments = pd.read_csv(os.path.join(data_path, 'olist_order_payments_dataset.csv'))

# 按 order_id 聚合
payments_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),          # 支付总金额
    payment_installments=('payment_installments', 'max'), # 最大分期数
    payment_type=('payment_type', lambda x: x.mode()[0])  # 取使用最多的支付方式
).reset_index()

orders = orders.merge(payments_agg, on='order_id', how='left')
print(f"合并支付后: {orders.shape}")

合并支付后: (96478, 17)


In [7]:
# 读取商品明细表
items = pd.read_csv(os.path.join(data_path, 'olist_order_items_dataset.csv'))

# 按 order_id 聚合
items_agg = items.groupby('order_id').agg(
    price_sum=('price', 'sum'),                 # 商品总价
    freight_sum=('freight_value', 'sum'),       # 总运费
    item_count=('order_item_id', 'count')       # 商品件数
).reset_index()

orders = orders.merge(items_agg, on='order_id', how='left')
print(f"合并商品后: {orders.shape}")

合并商品后: (96478, 20)


In [8]:
# 订单总额 = 商品总价 + 运费
orders['total_amount'] = orders['price_sum'] + orders['freight_sum']

# 检查是否有缺失值（可能有的订单没有商品记录，但已被我们过滤为 delivered，通常极少）
print(orders[['price_sum', 'freight_sum', 'total_amount']].isnull().sum())

price_sum       0
freight_sum     0
total_amount    0
dtype: int64


In [9]:
# 验证订单ID是否唯一（防止合并时数据膨胀）
print(f"宽表行数: {orders.shape[0]}")
print(f"唯一订单数: {orders['order_id'].nunique()}")
# 如果两者相等，说明合并完美，没有产生数据膨胀

# 保存到 processed 文件夹
orders.to_csv(os.path.join(processed_path, 'orders_master.csv'), index=False)
print("订单宽表已保存至 data/processed/orders_master.csv")

宽表行数: 96478
唯一订单数: 96478
订单宽表已保存至 data/processed/orders_master.csv
